# B2-019-attention-transformers — Practice p18 — Solution

**Type:** integrative · **Difficulty:** core · **Concepts:** scaled-dot-product-attention, attention-mask

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

Three independent probes expose the defects. (i) With q=k=(1,1), d_k=2, and a second zero key, scaled and unscaled first weights differ. (ii) With scores (0,0), post-softmax masking gives (1/2,0), whose row sum is 1/2; pre-softmax masking gives (1,0). (iii) For Nq=1 and Nk=2, Q @ K.T has shape (1,2), whereas K @ Q.T has the transposed shape (2,1). The corrected contract is: validate finite compatible arrays; scores=Q@K.T/sqrt(d_k); apply a Boolean allowed mask to scores; stable row softmax on keys; output=weights@V.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 1e-12
RTOL = 1e-12

q_scale = np.array([[1.0, 1.0]], dtype=np.float64)
k_scale = np.array([[1.0, 1.0], [0.0, 0.0]], dtype=np.float64)
raw = q_scale @ k_scale.T
scaled = raw / np.sqrt(2.0)
def softmax(x):
    shifted = x - np.max(x, axis=-1, keepdims=True)
    numer = np.exp(shifted)
    return numer / np.sum(numer, axis=-1, keepdims=True)
unscaled_weights = softmax(raw)
scaled_weights = softmax(scaled)

two_scores = np.array([[0.0, 0.0]], dtype=np.float64)
allowed = np.array([[True, False]])
postmasked_weights = softmax(two_scores) * allowed
premasked_weights = softmax(np.where(allowed, two_scores, -np.inf))

q_shape = np.array([[1.0, 0.0]], dtype=np.float64)
k_shape = np.array([[1.0, 0.0], [0.0, 1.0]], dtype=np.float64)
correct_scores = q_shape @ k_shape.T
reversed_scores = k_shape @ q_shape.T

v = np.array([[2.0, -1.0], [7.0, 5.0]], dtype=np.float64)
output = premasked_weights @ v

### Answer check

In [ ]:
assert not np.allclose(unscaled_weights, scaled_weights, atol=ATOL, rtol=RTOL)
assert np.isclose(postmasked_weights.sum(), 0.5, atol=ATOL, rtol=RTOL)
assert np.count_nonzero(premasked_weights[~allowed]) == 0
np.testing.assert_allclose(premasked_weights.sum(axis=-1), [1.0], atol=ATOL, rtol=RTOL)
assert np.all(np.isfinite(premasked_weights))
assert correct_scores.shape == (1, 2) and reversed_scores.shape == (2, 1)
assert output.shape == (1, 2)
assert np.all(np.isfinite(output))